In [1]:
!pip install -qU langchain langchain-community langchain-groq langchain-classic wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from langchain_groq import ChatGroq
from langchain_classic.agents import initialize_agent, AgentType, Tool
from langchain_classic.agents import AgentExecutor
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [3]:
# --- Set up the language model (the agent's brain) ---
llm = ChatGroq(
    api_key="", # enter your api key from groq
    model="llama-3.1-8b-instant",
    temperature=0
)

In [4]:
# --- Tool 1: Wikipedia ---
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

In [5]:
# --- Tool 2: a simple, safe calculator ---
def calculator(expression: str) -> str:
    """Evaluate a basic maths expression like '1879 * 2'."""
    try:
        # only allow numbers and maths symbols for safety
        allowed = set('0123456789+-*/(). ')
        if not set(expression).issubset(allowed):
            return 'Error: only numbers and + - * / ( ) are allowed.'
        return str(eval(expression))
    except Exception as e:
        return f'Error: {e}'

In [6]:
# --- Register both tools in a list ---
tools = [
    Tool(name='Wikipedia', func=wiki.run,
         description='Look up facts about people, places, and events.'),
    Tool(name='Calculator', func=calculator,
         description='Do arithmetic. Input a maths expression like 12*7.'),
]

In [7]:
# --- Build the ReAct agent ---
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True   # shows the step-by-step reasoning
)

/tmp/ipykernel_1363/707253275.py:2: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(


In [8]:
# --- Ask a multi-step factual question ---
question = (
    'Who developed the theory of relativity, '
    'and what is his birth year multiplied by 2?'
)


answer = agent.invoke({'input': question})
print('\nFINAL ANSWER:', answer['output'])



> Entering new AgentExecutor chain...
Thought: I need to find out who developed the theory of relativity and perform a calculation.
Action: Wikipedia
Action Input: theory of relativity
Observation: Page: Theory of relativity
Summary: The theory of relativity comprises two physics theories by Albert Einstein: special relativity and general relativity, proposed and published in 1905 and 1915, respectively. Special relativity applies to all physical phenomena in the absence of gravity. General relativity explains the law of gravitation and its relation to the forces of nature. It applies to the cosmological and astrophysical realm, including astronomy.
The theory transformed theoretical physics and astronomy during the 20th century, superseding a 200-year-old theory of mechanics created primarily by Isaac Newton. It introduced concepts including 4-dimensional spacetime as a unified entity of space and time, relativity of simultaneity, kinematic and gravitational time dilation, and lengt